# RAG with SQLite Backend

Swaps the in-memory `minsearch` index for the persistent `sqlitesearch` index built in `03-sqlite-ingest`. `RAGBase` accepts any index that implements `.search()`, so only the index construction changes — the RAG pipeline itself is identical.

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from zoomcamp.ingest import load_faq_data, build_index
from zoomcamp.rag_helper import RAGBase
from sqlitesearch import TextSearchIndex

In [2]:
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="db/faq.db"
)

In [3]:
sqlite_index.count()

243

In [4]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'I missed the first homework - can I still get a certificate?']

The index contains 158 documents (79 LLM Zoomcamp docs ingested twice — once from each index.add call during ingestion). The duplicate results are a known artifact of the SQLite index setup; the RAG output is unaffected.

In [5]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI(api_key=api_key)

In [6]:
assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [7]:
answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still open.
